# 1. Start with a LlamaIndex workflow

A workflow connects functions through **typed events**. A step receives an event, does some work, and returns the event another step expects. This notebook starts with one LLM call, then runs two calls concurrently and joins their results.

Set your chat provider in `.env` first. This notebook needs neither GoodMem nor local embeddings.

In [1]:
from llama_index.core.workflow import Context, Event, StartEvent, StopEvent, Workflow, step
from goodmem_rag.config import chat_model
from goodmem_rag.agents import chat

model = chat_model()

class ExplainWorkflow(Workflow):
    @step
    async def explain(self, ev: StartEvent) -> StopEvent:
        explanation = await chat(model, "Explain the topic in two sentences.", ev.topic)
        return StopEvent(result=explanation)

print(await ExplainWorkflow(timeout=120).run(topic="retrieval-augmented generation"))

Retrieval-augmented generation (RAG) is a technique that combines information retrieval with text generation, enabling models to fetch relevant data from external sources and use it to produce more accurate and contextually informed responses. This approach enhances the capabilities of language models by grounding their outputs in factual, up-to-date information rather than relying solely on pre-trained knowledge.


## Fan out, then join

The start step sends two events. LlamaIndex can run the two receiving steps concurrently. `collect_events` waits until both answers arrive, so the final step has a complete pair.

Each run has its own `Context`; it holds the events and state for that run.

In [2]:
class ExplainEvent(Event):
    topic: str

class ExampleEvent(Event):
    topic: str

class Explanation(Event):
    text: str

class Example(Event):
    text: str

class ParallelWorkflow(Workflow):
    @step
    async def start(self, ctx: Context, ev: StartEvent) -> ExplainEvent | ExampleEvent:
        ctx.send_event(ExplainEvent(topic=ev.topic))
        return ExampleEvent(topic=ev.topic)

    @step
    async def explain(self, ev: ExplainEvent) -> Explanation:
        return Explanation(text=await chat(model, "Explain the topic in two sentences.", ev.topic))

    @step
    async def example(self, ev: ExampleEvent) -> Example:
        return Example(text=await chat(model, "Give one concrete example in two sentences.", ev.topic))

    @step
    async def join(self, ctx: Context, ev: Explanation | Example) -> StopEvent | None:
        pair = ctx.collect_events(ev, [Explanation, Example])
        if pair is None:
            return None
        return StopEvent(result={"explanation": pair[0].text, "example": pair[1].text})

result = await ParallelWorkflow(timeout=120).run(topic="agentic RAG")
print(result["explanation"])
print(result["example"])

Agentic RAG (Retrieval-Augmented Generation) is an advanced AI framework that combines information retrieval with generative models, enabling systems to dynamically fetch relevant data from external sources and use it to generate contextually accurate and informed responses. This approach enhances the model's ability to handle complex queries by grounding its outputs in up-to-date or domain-specific information, making it particularly useful for tasks requiring factual accuracy and adaptability.
An example of **agentic RAG** is a system that, when asked to analyze a complex legal document, autonomously retrieves relevant case law, summarizes key points, and drafts a concise legal opinion, all while iteratively refining its search and reasoning based on the evolving context. This demonstrates its ability to act as an agent by dynamically planning, executing, and adapting its retrieval and generation processes to deliver a high-quality, context-aware response.


Try changing the topic. The two branches have different instructions but receive the same input. In notebook 2, only the branch selected by the model will run.